In [20]:
# ============================================
# BLOCK 1: IMPORTS AND SETUP
# ============================================
# WHAT THIS DOES:
# - Imports all necessary libraries
# - Sets up visualization style
# - Checks if SMOTE is available
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Check if SMOTE is available
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("✅ SMOTE is available!")
except ImportError:
    SMOTE_AVAILABLE = False
    print("⚠️ SMOTE not available. Will use class weights instead.")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*60)
print("FEATURE ENGINEERING NOTEBOOK")
print("="*60)
print(f"SMOTE Available: {SMOTE_AVAILABLE}")

✅ SMOTE is available!

FEATURE ENGINEERING NOTEBOOK
SMOTE Available: True


In [21]:
# ============================================
# BLOCK 2: LOAD THE DATASET
# ============================================
# WHAT THIS DOES:
# - Loads the dataset from the data folder
# - Creates a copy to preserve original data
# - Shows initial shape and columns
# ============================================

# Try multiple possible file paths
possible_paths = [
    '../data/CVD Dataset.csv',
    '../data/CVD_Dataset.csv',
    'data/CVD Dataset.csv',
    'data/CVD_Dataset.csv',
]

file_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        print(f"✅ Found file at: {path}")
        df = pd.read_csv(path)
        file_loaded = True
        break

if not file_loaded:
    raise FileNotFoundError("Could not find CVD Dataset.csv in any expected location!")

# Create a copy to work with
df_original = df.copy()
df = df.copy()

print("\n" + "="*60)
print("DATASET LOADED SUCCESSFULLY!")
print("="*60)
print(f"📊 Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"📋 Columns: {df.columns.tolist()}")

✅ Found file at: ../data/CVD Dataset.csv

DATASET LOADED SUCCESSFULLY!
📊 Shape: 1529 rows, 22 columns
📋 Columns: ['Sex', 'Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Blood Pressure (mmHg)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Height (cm)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Blood Pressure Category', 'Estimated LDL (mg/dL)', 'CVD Risk Score', 'CVD Risk Level']


In [ ]:
# ============================================
# BLOCK 3: HANDLE MISSING VALUES
# ============================================
print('\n' + '='*60)
print('HANDLING MISSING VALUES')
print('='*60)

missing_before = df.isnull().sum().sum()
print(f'\nMissing values BEFORE: {missing_before}')

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = ['Sex', 'Smoking Status', 'Diabetes Status',
                    'Physical Activity Level', 'Family History of CVD']

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

missing_after = df.isnull().sum().sum()
print(f'Missing values AFTER: {missing_after}')
print('All missing values handled')


In [23]:
# ============================================
# BLOCK 4: REMOVE REDUNDANT COLUMNS
# ============================================
# WHAT THIS DOES:
# - Removes Height (cm) because we have Height (m)
# - Removes Estimated LDL because it's derived from Total Cholesterol
# - KEEPS Waist-to-Height Ratio (clinically important!)
# ============================================

print("\n" + "="*60)
print("REMOVING REDUNDANT COLUMNS")
print("="*60)

print("\n❌ Columns to REMOVE:")
print("   1. Height (cm) - same as Height (m)")
print("   2. Estimated LDL (mg/dL) - derived from Total Cholesterol")

# Columns to drop
columns_to_drop = ['Height (cm)', 'Estimated LDL (mg/dL)']

# Check if columns exist before dropping
for col in columns_to_drop:
    if col in df.columns:
        print(f"   ✅ Removing: {col}")
        df = df.drop(columns=[col])
    else:
        print(f"   ⚠️ Column not found: {col}")

print(f"\n✅ Remaining columns: {len(df.columns)}")


REMOVING REDUNDANT COLUMNS

❌ Columns to REMOVE:
   1. Height (cm) - same as Height (m)
   2. Estimated LDL (mg/dL) - derived from Total Cholesterol
   ✅ Removing: Height (cm)
   ✅ Removing: Estimated LDL (mg/dL)

✅ Remaining columns: 20


In [24]:
# ============================================
# BLOCK 5: CREATE NEW CLINICAL FEATURES
# ============================================
# WHAT THIS DOES:
# - Creates PULSE PRESSURE (Systolic - Diastolic)
# - Creates CHOLESTEROL RATIO (Total Cholesterol / HDL)
# - Creates BMI CATEGORY (Underweight, Normal, Overweight, Obese)
# - Creates AGE GROUP (Young, Adult, Middle, Senior, Elderly)
# - Creates OBESITY RISK (1 if Obese, else 0)
# - Creates HYPERTENSION RISK (1 if High BP, else 0)
# - Creates METABOLIC SCORE (sum of risk factors)
# - Creates PHYSICAL ACTIVITY SCORE (numeric conversion)
# ============================================

print("\n" + "="*60)
print("CREATING NEW FEATURES")
print("="*60)

# 1. Pulse Pressure
print("\n🔧 Creating: Pulse Pressure")
df['Pulse Pressure'] = df['Systolic BP'] - df['Diastolic BP']
print(f"   ✅ Added: Pulse Pressure (mean: {df['Pulse Pressure'].mean():.1f})")

# 2. Cholesterol Ratio
print("\n🔧 Creating: Cholesterol Ratio")
df['Cholesterol_HDL_Ratio'] = df['Total Cholesterol (mg/dL)'] / df['HDL (mg/dL)']
print(f"   ✅ Added: Cholesterol_HDL_Ratio (mean: {df['Cholesterol_HDL_Ratio'].mean():.1f})")

# 3. BMI Category
print("\n🔧 Creating: BMI Category")
def get_bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_Category'] = df['BMI'].apply(get_bmi_category)
print(f"   ✅ Added: BMI_Category")
print(f"      Distribution:\n{df['BMI_Category'].value_counts()}")

# 4. Age Group
print("\n🔧 Creating: Age Group")
def get_age_group(age):
    if age < 35:
        return 'Young'
    elif age < 50:
        return 'Adult'
    elif age < 60:
        return 'Middle'
    elif age < 70:
        return 'Senior'
    else:
        return 'Elderly'

df['Age_Group'] = df['Age'].apply(get_age_group)
print(f"   ✅ Added: Age_Group")
print(f"      Distribution:\n{df['Age_Group'].value_counts()}")

# 5. Obesity Risk
print("\n🔧 Creating: Obesity Risk")
df['Obesity_Risk'] = (df['BMI'] >= 30).astype(int)
print(f"   ✅ Added: Obesity_Risk (Obese: {df['Obesity_Risk'].sum()} patients)")

# 6. Hypertension Risk
print("\n🔧 Creating: Hypertension Risk")
df['Hypertension_Risk'] = ((df['Systolic BP'] >= 140) | (df['Diastolic BP'] >= 90)).astype(int)
print(f"   ✅ Added: Hypertension_Risk (Hypertensive: {df['Hypertension_Risk'].sum()} patients)")

# 7. Metabolic Score
print("\n🔧 Creating: Metabolic Score")
df['Metabolic_Score'] = (
    df['Obesity_Risk'] + 
    df['Hypertension_Risk'] + 
    (df['Total Cholesterol (mg/dL)'] >= 240).astype(int) +
    (df['Fasting Blood Sugar (mg/dL)'] >= 126).astype(int) +
    (df['Smoking Status'] == 'Y').astype(int)
)
print(f"   ✅ Added: Metabolic_Score (mean: {df['Metabolic_Score'].mean():.1f})")

# 8. Physical Activity Score
print("\n🔧 Creating: Physical Activity Score")
activity_map = {'Low': 0, 'Moderate': 1, 'High': 2}
df['Physical_Activity_Score'] = df['Physical Activity Level'].map(activity_map)
print(f"   ✅ Added: Physical_Activity_Score")

# 9. Blood Pressure Category (derive if missing)
print("\n🔧 Creating/Verifying: Blood Pressure Category")
def get_bp_category(systolic, diastolic):
    if systolic < 120 and diastolic < 80:
        return 'Normal'
    elif systolic < 130 and diastolic < 80:
        return 'Elevated'
    elif systolic < 140 or diastolic < 90:
        return 'Hypertension Stage 1'
    else:
        return 'Hypertension Stage 2'

if 'Blood Pressure Category' not in df.columns or df['Blood Pressure Category'].isnull().sum() > 0:
    df['Blood Pressure Category'] = df.apply(
        lambda row: get_bp_category(row['Systolic BP'], row['Diastolic BP']), axis=1
    )
    print(f"   ✅ Added/Updated: Blood Pressure Category")
else:
    print(f"   ✅ Already exists: Blood Pressure Category")

print(f"\n📊 Blood Pressure Category Distribution:")
print(df['Blood Pressure Category'].value_counts())

print(f"\n✅ Total new features added: 9")


CREATING NEW FEATURES

🔧 Creating: Pulse Pressure
   ✅ Added: Pulse Pressure (mean: 42.7)

🔧 Creating: Cholesterol Ratio
   ✅ Added: Cholesterol_HDL_Ratio (mean: 3.8)

🔧 Creating: BMI Category
   ✅ Added: BMI_Category
      Distribution:
BMI_Category
Obese          635
Normal         421
Overweight     379
Underweight     94
Name: count, dtype: int64

🔧 Creating: Age Group
   ✅ Added: Age_Group
      Distribution:
Age_Group
Adult      676
Middle     369
Young      268
Senior     126
Elderly     90
Name: count, dtype: int64

🔧 Creating: Obesity Risk
   ✅ Added: Obesity_Risk (Obese: 635 patients)

🔧 Creating: Hypertension Risk
   ✅ Added: Hypertension_Risk (Hypertensive: 748 patients)

🔧 Creating: Metabolic Score
   ✅ Added: Metabolic_Score (mean: 2.1)

🔧 Creating: Physical Activity Score
   ✅ Added: Physical_Activity_Score

🔧 Creating/Verifying: Blood Pressure Category
   ✅ Already exists: Blood Pressure Category

📊 Blood Pressure Category Distribution:
Blood Pressure Category
Hyperten

In [ ]:
# ============================================
# BLOCK 6: ENCODE CATEGORICAL VARIABLES
# FIX: Drop original columns after encoding (no duplicates)
# ============================================
print('\n' + '='*60)
print('ENCODING CATEGORICAL VARIABLES')
print('='*60)

le = LabelEncoder()

# Binary encoding - create _Encoded columns
print('\nBinary Encoding:')
binary_cols = ['Sex', 'Smoking Status', 'Diabetes Status', 'Family History of CVD']
for col in binary_cols:
    if col in df.columns:
        df[f'{col}_Encoded'] = le.fit_transform(df[col].astype(str))
        print(f'  {col}: {df[col].unique()} -> {df[col + "_Encoded"].unique()}')

# Ordinal encoding
print('\nOrdinal Encoding:')
activity_map = {'Low': 0, 'Moderate': 1, 'High': 2}
df['Physical_Activity_Encoded'] = df['Physical Activity Level'].map(activity_map)
print('  Physical Activity Level: Low=0, Moderate=1, High=2')

bp_map = {'Normal': 0, 'Elevated': 1, 'Hypertension Stage 1': 2, 'Hypertension Stage 2': 3}
df['Blood_Pressure_Encoded'] = df['Blood Pressure Category'].map(bp_map)
print('  Blood Pressure Category: Normal=0, Elevated=1, Stage1=2, Stage2=3')

# Target: binary INTERMEDIARY=0, HIGH=1
print('\nTarget Encoding (binary):')
df['CVD_Risk_Encoded'] = df['CVD Risk Level'].map({'INTERMEDIARY': 0, 'HIGH': 1})
print('  CVD Risk Level: INTERMEDIARY=0, HIGH=1')
print(df['CVD_Risk_Encoded'].value_counts().to_string())

# One-hot encoding
df = pd.get_dummies(df, columns=['BMI_Category', 'Age_Group'], prefix=['BMI', 'Age'])
print('\nOne-hot encoded: BMI_Category, Age_Group')

# FIX: Drop original string columns to avoid duplicates in feature set
cols_to_drop = ['Sex', 'Smoking Status', 'Diabetes Status', 'Family History of CVD',
                'Physical Activity Level', 'Blood Pressure (mmHg)', 'Blood Pressure Category',
                'CVD Risk Level']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f'\nDropped original string columns (no duplicates now)')
print(f'Total features (excl. target): {len(df.columns) - 1}')


In [26]:
# ============================================
# BLOCK 7: SCALE NUMERICAL FEATURES
# ============================================
# WHAT THIS DOES:
# - Normalizes numerical features to same scale (mean=0, std=1)
# - Uses StandardScaler: (x - mean) / standard deviation
# ============================================

print("\n" + "="*60)
print("SCALING NUMERICAL FEATURES")
print("="*60)

# Identify numerical columns to scale
scale_cols = [
    'Age', 'Weight (kg)', 'Height (m)', 'BMI', 
    'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 
    'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 
    'Systolic BP', 'Diastolic BP', 'CVD Risk Score',
    'Pulse Pressure', 'Cholesterol_HDL_Ratio', 'Metabolic_Score',
    'Physical_Activity_Score'
]

# Filter columns that exist
scale_cols = [col for col in scale_cols if col in df.columns]

print(f"\n📊 Scaling {len(scale_cols)} numerical features...")

# Initialize scaler
scaler = StandardScaler()

# Scale the features
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])

print("\n📊 Before Scaling (Age example):")
print(f"   Mean: {df['Age'].mean():.2f}, Std: {df['Age'].std():.2f}")
print("\n📊 After Scaling (Age example):")
print(f"   Mean: {df_scaled['Age'].mean():.2f}, Std: {df_scaled['Age'].std():.2f}")

# Use scaled version
df = df_scaled
print("\n✅ Features scaled successfully!")


SCALING NUMERICAL FEATURES

📊 Scaling 15 numerical features...

📊 Before Scaling (Age example):
   Mean: 46.97, Std: 12.10

📊 After Scaling (Age example):
   Mean: 0.00, Std: 1.00

✅ Features scaled successfully!


In [ ]:
# ============================================
# BLOCK 8: TRAIN/TEST SPLIT THEN SMOTE
# FIX: Split first, apply SMOTE only to training data
# ============================================
print('\n' + '='*60)
print('TRAIN/TEST SPLIT THEN SMOTE (CORRECT ORDER)')
print('='*60)
print('NOTE: SMOTE is applied ONLY to training data.')
print('Test set uses original (non-synthetic) samples only.')
print('This prevents data leakage from synthetic samples.\n')

feature_cols = [col for col in df.columns if col != 'CVD_Risk_Encoded']
X = df[feature_cols]
y = df['CVD_Risk_Encoded']

# Split FIRST
from sklearn.model_selection import train_test_split
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size (original): {X_train_raw.shape[0]}')
print(f'Test size (original):  {X_test.shape[0]}')
print(f'Test target distribution: {y_test.value_counts().to_dict()}')

# Apply SMOTE to training split only
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_raw, y_train_raw)
print(f'\nAfter SMOTE - Train size: {X_resampled.shape[0]}')
print(f'After SMOTE - Train distribution: {pd.Series(y_resampled).value_counts().to_dict()}')
print('Block 8 complete')


In [ ]:
# ============================================
# BLOCK 9: SAVE PROCESSED DATA
# ============================================
print('\n' + '='*60)
print('SAVING PROCESSED DATA')
print('='*60)

processed_dir = '../data/processed/'

# Save original processed (scaled, no SMOTE)
df.to_csv(f'{processed_dir}processed_data_original.csv', index=False)
print(f'Saved: processed_data_original.csv ({df.shape[0]} rows)')

# Save clean (same as original for this pipeline)
df.to_csv(f'{processed_dir}processed_data_clean.csv', index=False)
print(f'Saved: processed_data_clean.csv ({df.shape[0]} rows)')

# Save SMOTE data (training set after SMOTE, for reference)
df_smote_save = pd.DataFrame(X_resampled, columns=feature_cols)
df_smote_save['CVD_Risk_Encoded'] = y_resampled
df_smote_save.to_csv(f'{processed_dir}processed_data_smote.csv', index=False)
print(f'Saved: processed_data_smote.csv ({df_smote_save.shape[0]} train rows, SMOTE applied)')

import joblib
joblib.dump(scaler, f'{processed_dir}scaler.pkl')
print(f'Saved: scaler.pkl')

print('\n' + '='*60)
print('FEATURE ENGINEERING COMPLETE')
print('='*60)
print('\nKey corrections applied:')
print('  1. LOW risk class excluded (binary classification: HIGH vs INTERMEDIARY)')
print('  2. Original string columns dropped after encoding (no duplicate features)')
print('  3. SMOTE applied only to training split (no data leakage)')
print('\nReady for Model Building')
